# Lab 6 · Serving A — Ollama + GGUF

**~20 minutes.**

The GGUF from Lab 5 becomes a running model you can chat with, plus an
OpenAI-compatible API endpoint.

This lab also brings back every sampling parameter from Lab 1 — but as
**configuration** rather than Python arguments, which is how you'd
actually ship them.

> ↳ Slides: *Model Format: GGUF...* · *Inference Parameters*

In [ ]:
# --- Locate the workshop repo -------------------------------------------
# Tries, in order: already present -> attached Kaggle Dataset -> git clone.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Ankush610/LLM-lab.git"

def find_repo() -> Path:
    for candidate in [Path("/kaggle/working/LLM-lab"), Path.cwd(), Path.cwd().parent]:
        if (candidate / "common" / "config.py").exists():
            return candidate
    for d in Path("/kaggle/input").glob("*"):          # attached as a Dataset
        if (d / "common" / "config.py").exists():
            return d
    print("Repo not found locally, cloning...")        # last resort
    subprocess.run(["git", "clone", "-q", REPO_URL, "/kaggle/working/LLM-lab"], check=True)
    return Path("/kaggle/working/LLM-lab")

REPO = find_repo()
sys.path[:0] = [str(REPO / "common"), str(REPO / "dataset")]
print(f"repo: {REPO}")

import config
print(config.summary())

## 7.1 Installing Ollama on Kaggle

Ollama's installer normally wants root and systemd. On Kaggle:

- **root** — we have it. Kaggle notebook sessions run as root (`whoami`
  below proves it), so the install script works unmodified.
- **systemd** — not running. The installer warns and skips service setup,
  so we start the server ourselves as a background process.

Worth internalising: *"needs root"* usually means *"needs to write to
`/usr`"*. On a cluster where you have no root, the same binary works fine
extracted into your own directory — which is exactly what the cluster
version of this workshop does.

In [ ]:
!whoami
!curl -fsSL https://ollama.com/install.sh | sh

### Start the server

No systemd, so we launch it ourselves and poll until the port answers.
Polling rather than `sleep` — the startup time varies.

In [ ]:
import subprocess, time, os, requests

os.environ["OLLAMA_HOST"] = f"127.0.0.1:{config.OLLAMA_PORT}"
os.environ["OLLAMA_MODELS"] = "/kaggle/working/ollama_models"   # not $HOME

server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/kaggle/working/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)

url = f"http://127.0.0.1:{config.OLLAMA_PORT}"
for attempt in range(60):
    try:
        if requests.get(url, timeout=2).status_code == 200:
            print(f"ollama up after {attempt+1}s")
            break
    except requests.exceptions.RequestException:
        time.sleep(1)
else:
    print("ollama did not start - check /kaggle/working/ollama.log")
    !tail -20 /kaggle/working/ollama.log

## 7.2 The Modelfile

A `Modelfile` is Ollama's packaging format — the model file plus its
default behaviour, in one place.

Everything from Lab 1's playground appears here as a `PARAMETER`. That is
the point of this lab: the sampling knobs aren't a Python detail, they're
deployment configuration you ship with the model.

| directive | what it does |
|---|---|
| `FROM` | the GGUF file |
| `TEMPLATE` | the chat template — **third time you've seen this** |
| `PARAMETER` | default sampling settings |
| `SYSTEM` | the baked-in system prompt |

The `TEMPLATE` must match what we trained with. Get it wrong and quality
degrades quietly — the same failure as Lab 2, in a different costume.

In [ ]:
gguf_files = sorted(config.GGUF_DIR.glob("*.gguf"))
assert gguf_files, f"No GGUF in {config.GGUF_DIR} - re-run Lab 5."
GGUF_PATH = gguf_files[0]
print(f"using {GGUF_PATH.name}  ({GGUF_PATH.stat().st_size/2**30:.2f} GB)")

In [ ]:
# The TEMPLATE must match the model we actually trained. Llama 3 and
# Qwen/ChatML use different special tokens, so pick by config - otherwise
# anyone on the ungated fallback gets a silently wrong template.
if "llama" in config.CHAT_TEMPLATE:
    template = (
        "<|start_header_id|>system<|end_header_id|>\n\n"
        "{{ .System }}<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        "{{ .Prompt }}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
    )
    stops = ["<|eot_id|>", "<|start_header_id|>"]
else:
    template = (
        "<|im_start|>system\n{{ .System }}<|im_end|>\n"
        "<|im_start|>user\n{{ .Prompt }}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )
    stops = ["<|im_end|>", "<|im_start|>"]

q3 = chr(34) * 3        # triple quote, kept out of the f-string below
modelfile = "\n".join([
    f"FROM {GGUF_PATH}",
    "",
    f"TEMPLATE {q3}{template}{q3}",
    "",
    f"SYSTEM {q3}{config.SYSTEM_PROMPT}{q3}",
    "",
    f"PARAMETER temperature    {config.TEMPERATURE}",
    f"PARAMETER top_k          {config.TOP_K}",
    f"PARAMETER top_p          {config.TOP_P}",
    f"PARAMETER repeat_penalty {config.REPETITION_PENALTY}",
    f"PARAMETER num_ctx        {config.MAX_SEQ_LENGTH}",
    *[f'PARAMETER stop           "{s}"' for s in stops],
])

open("/kaggle/working/Modelfile", "w").write(modelfile)
print(modelfile)

## 7.3 Build and run

`ollama create` reads the Modelfile and registers the model. Takes a
minute or two — it copies the GGUF into Ollama's store.

In [ ]:
!ollama create {config.OLLAMA_MODEL_NAME} -f /kaggle/working/Modelfile
!ollama list

In [ ]:
# Your model, running under its own name.
!ollama run {config.OLLAMA_MODEL_NAME} "Which partition should I use for a 4-hour A100 job on AURA?"

## 7.4 The OpenAI-compatible endpoint

Ollama exposes an OpenAI-shaped API on port 11434. Any code written
against the OpenAI SDK works by changing the `base_url` — which is why
this is the easy path from a notebook to an actual application.

In [ ]:
import requests, json

def chat(prompt, **params):
    r = requests.post(f"{url}/v1/chat/completions", json={
        "model": config.OLLAMA_MODEL_NAME,
        "messages": [
            {"role": "system", "content": config.SYSTEM_PROMPT},
            {"role": "user",   "content": prompt},
        ],
        **params,
    }, timeout=180)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

from eval_prompts import EVAL_PROMPTS
for p in EVAL_PROMPTS[:4]:
    print(f"Q: {p['question']}")
    print(f"A: {chat(p['question']).strip()[:300]}\n")

### The same parameters, now over HTTP

Second pass over the *Inference Parameters* section. Same concepts as Lab
1, different transport — and note the names differ slightly between the
HuggingFace API, the Modelfile, and the OpenAI API. That inconsistency is
a real and permanent annoyance.

| concept | HF `generate()` | Modelfile | OpenAI API |
|---|---|---|---|
| randomness | `temperature` | `temperature` | `temperature` |
| nucleus | `top_p` | `top_p` | `top_p` |
| top-k | `top_k` | `top_k` | *(not exposed)* |
| repetition | `repetition_penalty` | `repeat_penalty` | `frequency_penalty` / `presence_penalty` |
| length | `max_new_tokens` | `num_predict` | `max_tokens` |
| context | *(model config)* | `num_ctx` | *(server-side)* |

In [ ]:
q = "Explain in one sentence what the aura-debug QoS is for."

for t in (0.0, 0.8, 1.6):
    print(f"--- temperature={t} ---")
    print(chat(q, temperature=t, max_tokens=90).strip()[:260], "\n")

## 7.5 Did quantization hurt?

The Ollama model is `q4_k_m` — 4-bit. Lab 4's answers came from the
4-bit-base-plus-fp16-adapter version. Different numerics, same training.

Read a few side by side. For most chat use the difference is hard to
spot, which is the whole reason 4-bit quantization is the default choice
for local deployment.

> ↳ Slide: *What is Quantization*

In [ ]:
import compare
tuned = compare.load(config.TUNED_ANSWERS)     # from Lab 4

for p in EVAL_PROMPTS[:3]:
    print("=" * 72)
    print("Q:", p["question"])
    print("-" * 72)
    print("Lab 4 (4-bit + fp16 adapter):\n ", tuned[p["id"]].strip()[:280])
    print("\nOllama (q4_k_m GGUF):\n ", chat(p["question"]).strip()[:280])
    print()

## Take it home

The GGUF is a single self-contained file. Download it from the Kaggle
output panel, and on your own machine:

```bash
ollama create aura-support -f Modelfile
ollama run aura-support
```

No GPU required — llama.cpp runs a 3B q4 model on a laptop CPU at
readable speed. That's the practical appeal of this format.

---

### Next: `07_serve_vllm.ipynb` — vLLM, throughput, and serving adapters without merging

> **Kaggle tip:** if the session has been idle a while, check the right-hand
> panel still shows the GPU attached before starting the next notebook.